In [1]:
from pyspark.sql import SparkSession

In [2]:
from pyspark.sql.functions import explode,col,expr,array,when
from pyspark.sql.types import ArrayType, IntegerType, ShortType

In [3]:
spark = SparkSession.builder.appName("task").getOrCreate()

25/04/30 11:06:02 WARN Utils: Your hostname, milanthapa resolves to a loopback address: 127.0.1.1; using 10.10.42.122 instead (on interface enp2s0)
25/04/30 11:06:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/30 11:06:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
spark

In [5]:
df_pyspark = spark.read.option('multiline','true').json('provider.json')

In [6]:
df_pyspark.printSchema()

root
 |-- provider_group_id: long (nullable = true)
 |-- provider_groups: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- npi: array (nullable = true)
 |    |    |    |-- element: long (containsNull = true)
 |    |    |-- tin: struct (nullable = true)
 |    |    |    |-- type: string (nullable = true)
 |    |    |    |-- value: string (nullable = true)



In [7]:
provider_group = df_pyspark.withColumn("new_provider", explode("provider_groups"))
provider_new = provider_group.withColumn("npi",explode("new_provider.npi"))
provider_again = provider_new.select(
    "provider_group_id",
    col("npi").alias("npi"),
    col("new_provider.tin.type").alias("tin_type"),
    col("new_provider.tin.value").alias("tin")
)      

In [8]:
provider_again.printSchema()

root
 |-- provider_group_id: long (nullable = true)
 |-- npi: long (nullable = true)
 |-- tin_type: string (nullable = true)
 |-- tin: string (nullable = true)



In [9]:
provider_again.show()

+-----------------+----------+--------+----------+
|provider_group_id|       npi|tin_type|       tin|
+-----------------+----------+--------+----------+
|         10001001|1235233008|     ein|04-3267217|
|         10001001|1316041189|     ein|04-3267217|
|         10001001|1780788554|     ein|04-3267217|
|         10001001|1891068409|     ein|04-3267217|
|         10001001|1366459570|     ein|11-1562701|
|         10001001|1417915653|     ein|11-3358535|
|         10001001|1417915653|     ein|13-3888838|
|         10002001|1609829761|     ein|00-0004110|
|         10002001|1821482241|     ein|00-0004110|
|         10002001|1760986277|     ein|00-6980743|
|         10002001|1215075882|     ein|01-0550744|
|         10002001|1013917665|     ein|01-0555304|
|         10002001|1679780811|     ein|01-0555483|
|         10002001|1700093952|     ein|01-0555483|
|         10002001|1780072447|     ein|01-0555483|
|         10002001|1952532970|     ein|01-0555483|
|         10002001|1376647511| 

In [16]:
provider_tin=provider_again.withColumn("tin_type",when(col("tin_type")=="ein",1)
                                      .when(col("tin_type")=="npi",2))
provider_final = provider_tin.withColumn("tin",expr("REPLACE(tin,'-','')"))

In [17]:
provider_final.show()

+-----------------+----------+--------+---------+
|provider_group_id|       npi|tin_type|      tin|
+-----------------+----------+--------+---------+
|         10001001|1235233008|    NULL|043267217|
|         10001001|1316041189|    NULL|043267217|
|         10001001|1780788554|    NULL|043267217|
|         10001001|1891068409|    NULL|043267217|
|         10001001|1366459570|    NULL|111562701|
|         10001001|1417915653|    NULL|113358535|
|         10001001|1417915653|    NULL|133888838|
|         10002001|1609829761|    NULL|000004110|
|         10002001|1821482241|    NULL|000004110|
|         10002001|1760986277|    NULL|006980743|
|         10002001|1215075882|    NULL|010550744|
|         10002001|1013917665|    NULL|010555304|
|         10002001|1679780811|    NULL|010555483|
|         10002001|1700093952|    NULL|010555483|
|         10002001|1780072447|    NULL|010555483|
|         10002001|1952532970|    NULL|010555483|
|         10002001|1376647511|    NULL|010567880|


In [19]:
provider_cast = provider_final.withColumn("tin_type", col("tin_type").cast(ShortType()))

In [20]:
provider_cast.printSchema()

root
 |-- provider_group_id: long (nullable = true)
 |-- npi: long (nullable = true)
 |-- tin_type: short (nullable = true)
 |-- tin: string (nullable = true)



In [21]:
provider_cast.write.parquet('provider_output.parquet')